# PRUDENCIA — Entraînement du modèle Random Forest

Ce notebook présente, étape par étape, le pipeline Machine Learning utilisé dans le projet **PRUDENCIA**.

## Objectif

Prédire le niveau de risque AI Act d'un projet d'intelligence artificielle à partir de plusieurs variables descriptives :

- secteur d'activité ;
- rôle de l'organisation ;
- présence de données personnelles ;
- présence de données sensibles ;
- type de système d'intelligence artificielle.

Le modèle retenu est un **Random Forest Classifier**.

## 1. Import des bibliothèques

Les bibliothèques principales utilisées sont :

- `pandas` pour manipuler les données ;
- `scikit-learn` pour préparer, entraîner et évaluer le modèle ;
- `matplotlib` pour les graphiques ;
- `joblib` pour sauvegarder le pipeline entraîné.

In [1]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print('Bibliothèques importées avec succès.')

Bibliothèques importées avec succès.


## 2. Configuration générale

Cette cellule définit :

- la graine aléatoire pour obtenir des résultats reproductibles ;
- la part du dataset réservée au test ;
- la colonne cible ;
- les variables explicatives utilisées par le modèle.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.20

TARGET_COLUMN = 'risk_level_aiact'

FEATURE_COLUMNS = [
    'secteur_grp',
    'role',
    'donnees_perso',
    'donnees_sensibles',
    'type_ia_norm',
]

MODEL_OUTPUT_DIR = Path('outputs/random_forest')
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration terminée.')

## 3. Chargement du dataset

Modifier la variable `DATASET_PATH` pour indiquer l'emplacement réel du fichier CSV.

Le paramètre `sep=None` permet à pandas de détecter automatiquement si le séparateur est une virgule ou un point-virgule.

In [ ]:
DATASET_PATH = Path('./datasets/ml/ml_training_datasets_v1.csv')

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Dataset introuvable : {DATASET_PATH.resolve()}\n'
        'Modifie la variable DATASET_PATH avant de poursuivre.'
    )

df = pd.read_csv(DATASET_PATH, sep=None, engine='python')

print(f'Nombre de lignes : {len(df)}')
print(f'Nombre de colonnes : {len(df.columns)}')
df.head()

## 4. Inspection du dataset

Avant l'entraînement, il est important de vérifier :

- les noms des colonnes ;
- les types de données ;
- la présence éventuelle de valeurs manquantes ;
- la répartition des classes de la variable cible.

In [ ]:
print('Colonnes disponibles :')
print(df.columns.tolist())

print('\nTypes de données :')
display(df.dtypes.to_frame('type'))

print('\nValeurs manquantes :')
display(df.isna().sum().to_frame('valeurs_manquantes'))

In [ ]:
required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        'Colonnes obligatoires absentes : '
        + ', '.join(missing_columns)
    )

print('Toutes les colonnes obligatoires sont présentes.')

## 5. Nettoyage et préparation

Dans cette version simple du pipeline :

- seules les colonnes utiles sont conservées ;
- les lignes sans valeur cible sont supprimées ;
- les valeurs manquantes des variables explicatives sont remplacées par `inconnu` ;
- les variables catégorielles sont converties en texte.

In [ ]:
prepared_df = df[FEATURE_COLUMNS + [TARGET_COLUMN]].copy()

prepared_df = prepared_df.dropna(subset=[TARGET_COLUMN])

for column in FEATURE_COLUMNS:
    prepared_df[column] = (
        prepared_df[column]
        .fillna('inconnu')
        .astype(str)
        .str.strip()
        .replace('', 'inconnu')
    )

prepared_df[TARGET_COLUMN] = (
    prepared_df[TARGET_COLUMN]
    .astype(str)
    .str.strip()
)

print(f'Nombre de lignes après préparation : {len(prepared_df)}')
prepared_df.head()

In [ ]:
class_distribution = prepared_df[TARGET_COLUMN].value_counts()
display(class_distribution.to_frame('nombre_exemples'))

class_distribution.plot(kind='bar', figsize=(9, 5))
plt.title('Répartition des classes AI Act')
plt.xlabel('Classe')
plt.ylabel("Nombre d'exemples")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Séparation des variables et de la cible

- `X` contient les variables explicatives ;
- `y` contient la classe à prédire.

In [ ]:
X = prepared_df[FEATURE_COLUMNS]
y = prepared_df[TARGET_COLUMN]

print('Dimensions de X :', X.shape)
print('Dimensions de y :', y.shape)

## 7. Séparation Train / Test

Le modèle apprend sur l'ensemble d'entraînement.

L'ensemble de test est conservé à part afin d'évaluer le modèle sur des données qu'il n'a jamais vues.

La stratification est utilisée lorsque chaque classe possède au moins deux exemples.

In [ ]:
class_counts = y.value_counts()
can_stratify = len(class_counts) > 1 and class_counts.min() >= 2

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y if can_stratify else None,
)

print(f'Train : {len(X_train)} lignes')
print(f'Test  : {len(X_test)} lignes')
print(f'Stratification : {can_stratify}')

## 8. Encodage des variables catégorielles

Le modèle Random Forest travaille avec des valeurs numériques.

Le `OneHotEncoder` transforme chaque catégorie en colonne binaire.

L'option `handle_unknown='ignore'` permet au pipeline d'accepter une catégorie nouvelle lors d'une prédiction.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False,
            ),
            FEATURE_COLUMNS,
        )
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

print('Préprocesseur créé.')

## 9. Création du modèle Random Forest

Une Random Forest combine plusieurs arbres de décision.

Chaque arbre produit une prédiction, puis la forêt effectue un vote majoritaire.

Hyperparamètres utilisés :

- `n_estimators=100` : 100 arbres ;
- `max_depth=None` : profondeur non limitée ;
- `random_state=42` : résultats reproductibles ;
- `n_jobs=-1` : utilisation de tous les cœurs disponibles.

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', classifier),
    ]
)

pipeline

## 10. Entraînement du modèle

La méthode `fit()` réalise deux opérations :

1. le préprocesseur apprend les catégories présentes dans le jeu d'entraînement ;
2. le Random Forest apprend à associer les variables aux classes de risque.

In [ ]:
pipeline.fit(X_train, y_train)

print('Entraînement terminé.')

## 11. Prédictions sur le jeu de test

In [ ]:
y_pred = pipeline.predict(X_test)

predictions_df = X_test.copy()
predictions_df['classe_reelle'] = y_test.values
predictions_df['classe_predite'] = y_pred

predictions_df.head(10)

## 12. Évaluation du modèle

Les principales métriques sont :

- **Accuracy** : proportion globale de bonnes prédictions ;
- **Precision** : fiabilité des prédictions positives ;
- **Recall** : capacité à retrouver les exemples d'une classe ;
- **F1-score** : compromis entre précision et rappel.

La moyenne `macro` donne le même poids à chaque classe.

In [ ]:
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision_macro': precision_score(
        y_test, y_pred, average='macro', zero_division=0
    ),
    'recall_macro': recall_score(
        y_test, y_pred, average='macro', zero_division=0
    ),
    'f1_macro': f1_score(
        y_test, y_pred, average='macro', zero_division=0
    ),
    'f1_weighted': f1_score(
        y_test, y_pred, average='weighted', zero_division=0
    ),
}

metrics_df = pd.DataFrame.from_dict(
    metrics,
    orient='index',
    columns=['score'],
)

display(metrics_df)

In [ ]:
report = classification_report(
    y_test,
    y_pred,
    zero_division=0,
)

print(report)

## 13. Matrice de confusion

La matrice de confusion compare les classes réelles et les classes prédites.

Les valeurs situées sur la diagonale correspondent aux bonnes prédictions.

In [ ]:
labels = sorted(set(y_test.astype(str)) | set(pd.Series(y_pred).astype(str)))

matrix = confusion_matrix(
    y_test,
    y_pred,
    labels=labels,
)

confusion_df = pd.DataFrame(
    matrix,
    index=[f'réel_{label}' for label in labels],
    columns=[f'prédit_{label}' for label in labels],
)

display(confusion_df)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    xticks_rotation=45,
)

plt.title('Matrice de confusion - Random Forest')
plt.tight_layout()
plt.show()

## 14. Importance des variables

Le Random Forest calcule une importance pour chaque variable encodée.

Une importance élevée signifie que la variable a souvent été utile pour séparer les classes.

Attention : cela n'indique pas une relation de cause à effet.

In [ ]:
feature_names = (
    pipeline
    .named_steps['preprocessor']
    .get_feature_names_out()
)

feature_importances = (
    pipeline
    .named_steps['classifier']
    .feature_importances_
)

importance_df = (
    pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importances,
    })
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

importance_df.head(15)

In [ ]:
top_features = (
    importance_df
    .head(15)
    .sort_values('importance', ascending=True)
)

plt.figure(figsize=(10, 7))
plt.barh(top_features['feature'], top_features['importance'])
plt.title('Variables les plus importantes')
plt.xlabel('Importance')
plt.ylabel('Variable encodée')
plt.tight_layout()
plt.show()

## 15. Sauvegarde du pipeline et des résultats

Le pipeline sauvegardé contient à la fois :

- l'encodage des variables ;
- le modèle Random Forest entraîné.

Il pourra ensuite être rechargé par l'API PRUDENCIA pour effectuer des prédictions.

In [ ]:
model_path = MODEL_OUTPUT_DIR / 'random_forest_pipeline.joblib'
metrics_path = MODEL_OUTPUT_DIR / 'metrics.json'
report_path = MODEL_OUTPUT_DIR / 'classification_report.txt'
confusion_path = MODEL_OUTPUT_DIR / 'confusion_matrix.csv'
importance_path = MODEL_OUTPUT_DIR / 'feature_importances.csv'

joblib.dump(pipeline, model_path)

metrics_path.write_text(
    json.dumps(metrics, indent=4, ensure_ascii=False),
    encoding='utf-8',
)

report_path.write_text(report, encoding='utf-8')
confusion_df.to_csv(confusion_path, encoding='utf-8')
importance_df.to_csv(
    importance_path,
    index=False,
    encoding='utf-8',
)

print('Fichiers sauvegardés dans :', MODEL_OUTPUT_DIR.resolve())

## 16. Prédiction sur un nouveau projet

Le nouvel exemple doit contenir les mêmes variables que celles utilisées pendant l'entraînement.

Les valeurs présentées ci-dessous sont uniquement des exemples et doivent correspondre aux catégories présentes dans le dataset.

In [ ]:
new_project = pd.DataFrame([
    {
        'secteur_grp': 'sante',
        'role': 'fournisseur',
        'donnees_perso': 'oui',
        'donnees_sensibles': 'oui',
        'type_ia_norm': 'classification',
    }
])

predicted_class = pipeline.predict(new_project)[0]
probabilities = pipeline.predict_proba(new_project)[0]
classes = pipeline.named_steps['classifier'].classes_

result = {
    'predicted_risk_level': str(predicted_class),
    'probabilities': {
        str(label): float(probability)
        for label, probability in zip(classes, probabilities)
    },
}

print(json.dumps(result, indent=4, ensure_ascii=False))

## 17. Rechargement du modèle sauvegardé

Cette étape simule le comportement futur de l'API PRUDENCIA : le pipeline est chargé depuis le disque, puis utilisé sans nouvel entraînement.

In [ ]:
loaded_pipeline = joblib.load(model_path)

loaded_prediction = loaded_pipeline.predict(new_project)[0]

print('Prédiction du pipeline rechargé :', loaded_prediction)

## Conclusion

Ce notebook a présenté un pipeline Machine Learning complet :

1. chargement du dataset ;
2. contrôle et nettoyage ;
3. séparation train/test ;
4. encodage des catégories ;
5. entraînement d'un Random Forest ;
6. évaluation avec plusieurs métriques ;
7. analyse de la matrice de confusion ;
8. interprétation de l'importance des variables ;
9. sauvegarde et rechargement du pipeline ;
10. prédiction sur un nouveau projet.

Ce pipeline constitue la brique **Machine Learning** de PRUDENCIA.